# 🦟 DengueRadar — 1-Week-Ahead Risk Forecast

**Goal:** Predict, for every MOH area in Sri Lanka, the dengue risk tier (Low / Watch / Warning / Alert) and expected case count for the **next week** (week `t+1`).

**Stack:** 3 base learners (LightGBM, XGBoost, CatBoost) → Logistic Regression meta-learner (stacking).

**Why these models?**
- All three are gradient-boosted decision trees — they handle tabular features, missing values, and non-linear interactions naturally.
- They are fast to train, easy to retrain on new weekly data, and inference is a few ms per row.
- Stacking with a simple LR meta-learner usually beats any single model on tabular tasks and lets us down-weight a single weak model without losing the others.

**Why no LSTM?** The 4-model stack with LSTM was only ~0.2% better in offline eval, but adds a 12-week sequence preprocessing step at every inference call — bad for a weekly cron job. The 3-model stack is what ships to production.

**Notebook outline:**
1. Setup
2. Data loading
3. Target & splits (1-week-ahead)
4. Feature engineering (63 features, all lagged, no leakage)
5. Train base models (tier classifier + case-count regressor)
6. Stacking ensemble
7. Performance summary
8. Save artifacts
9. `DengueRadarPredictor` — production inference class
10. Web app integration (FastAPI example)
11. Deployment checklist

## 1. Setup

In [1]:
import subprocess, sys, os, json, pickle, warnings
warnings.filterwarnings("ignore")

IN_COLAB = False
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    pass

for pkg in ["lightgbm", "catboost", "xgboost", "pyarrow"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        except subprocess.CalledProcessError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--break-system-packages", pkg])

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import random

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.titleweight"] = "bold"

MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

# Risk tier definitions (order matters)
TIER_ORDER = ["Low", "Watch", "Warning", "Alert"]
TIER_TO_INT = {t: i for i, t in enumerate(TIER_ORDER)}
INT_TO_TIER = {i: t for t, i in TIER_TO_INT.items()}
ALERT_THRESHOLD = 0.5  # probability above which we call an Alert "high confidence"

print(f"  |  Python: {sys.version.split()[0]}  |  Colab: {IN_COLAB}")


Installing catboost...
  |  Python: 3.12.13  |  Colab: True


## 2. Data Loading

**Input file:** `dengueradar_training_table.csv` — one row per (MOH area, week).

**Required columns** (raw, before feature engineering):

| Column | Type | Description |
|---|---|---|
| `moh_name` | str | MOH area name (≈226 areas in Sri Lanka) |
| `district` | str | District name |
| `week_start` | date (YYYY-MM-DD) | Monday of the week |
| `cases` | int | Reported dengue cases that week |
| `incidence_per_100k` | float | Cases per 100k population |
| `population` | int | MOH population |
| `pop_density` | float | People per km² |
| `temp_avg`, `temp_max`, `temp_min` | float | Weekly average/max/min temperature (°C) |
| `humidity` | float | Weekly average humidity (%) |
| `rain_1w`, `rain_2w`, `rain_4w` | float | Rainfall in last 1/2/4 weeks |
| `temp_avg_4w`, `humidity_4w` | float | 4-week rolling averages |
| `risk_tier` | str | Original tier label (see note below) |

**Note on the original `risk_tier` column:** it was defined by percentile thresholds on the *entire* dataset, which is a form of target leakage. We **re-derive** the tier from `incidence_per_100k` using thresholds computed on the **training period only**.

In [2]:
if IN_COLAB:
    print("Upload dengueradar_training_table.csv ...")
    uploaded = files.upload()
    DATA_PATH = list(uploaded.keys())[0]
else:
    candidates = [
        "dengueradar_training_table.csv",
        "../dengueradar_training_table.csv",
        "/workspace/dengueradar_training_table.csv",
        "/content/dengueradar_training_table.csv",
    ]
    DATA_PATH = next((p for p in candidates if os.path.exists(p)), None)
    if DATA_PATH is None:
        raise FileNotFoundError(
            "Place dengueradar_training_table.csv in the current directory or set DATA_PATH."
        )
    print(f"Found data at: {DATA_PATH}")

df_raw = pd.read_csv(DATA_PATH)
df_raw["week_start"] = pd.to_datetime(df_raw["week_start"])
print(f"Loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"Date range: {df_raw['week_start'].min().date()} → {df_raw['week_start'].max().date()}")
print(f"MOH areas: {df_raw['moh_name'].nunique()}  |  Districts: {df_raw['district'].nunique()}")
df_raw.head()


Upload dengueradar_training_table.csv ...


Saving dengueradar_training_table.csv to dengueradar_training_table.csv
Loaded: 145,985 rows × 28 columns
Date range: 2013-12-28 → 2026-05-11
MOH areas: 226  |  Districts: 25


,moh_code,moh_name,ds_name,ds_id,district,week_start,iso_year,iso_week,cases,cases_lag1,...,rain_2w,rain_4w,temp_avg_4w,humidity_4w,population,pop_density,birth_rate,area_km2,centroid_lat,centroid_lon
0,10010010,Dehiwala,Dehiwala-Mount Lavinia,LKA.5.2_1,Colombo,2013-12-28,2013,52,13,NaN,...,2.07,2.07,24.930000,82.250000,89651.0,4080.217857,16.1,21.972111,6.839427,79.877068
1,10010010,Dehiwala,Dehiwala-Mount Lavinia,LKA.5.2_1,Colombo,2014-01-04,2014,1,10,13.0,...,29.89,29.89,25.119286,83.143571,89651.0,4080.217857,16.1,21.972111,6.839427,79.877068
2,10010010,Dehiwala,Dehiwala-Mount Lavinia,LKA.5.2_1,Colombo,2014-01-11,2014,2,16,10.0,...,61.10,63.17,25.219048,83.536190,89651.0,4080.217857,16.1,21.972111,6.839427,79.877068
3,10010010,Dehiwala,Dehiwala-Mount Lavinia,LKA.5.2_1,Colombo,2014-01-18,2014,3,11,16.0,...,33.47,63.36,25.257143,82.368214,89651.0,4080.217857,16.1,21.972111,6.839427,79.877068
4,10010010,Dehiwala,Dehiwala-Mount Lavinia,LKA.5.2_1,Colombo,2014-01-25,2014,4,8,11.0,...,12.88,73.98,25.331071,81.412857,89651.0,4080.217857,16.1,21.972111,6.839427,79.877068


## 3. Target & Splits (1-Week-Ahead)

**Key change vs the previous notebook:** the target is shifted by one week.

- `target_tier`  = `risk_tier_int` of week `t+1` for the same MOH
- `target_cases` = `cases`         of week `t+1` for the same MOH

Features for row `(MOH, week t)` are still computed from data **≤ t** (no leakage), but the model now predicts the *next* week, which is what the web app actually wants.

**Tier re-labeling** uses incidence-per-100k thresholds learned on the training period only:
- Low:      < 50th pct
- Watch:    50th–80th pct
- Warning:  80th–95th pct
- Alert:    ≥ 95th pct

In [3]:
df = df_raw.copy().sort_values(["moh_name", "week_start"]).reset_index(drop=True)

train_mask = df["week_start"] < "2024-01-01"
T1, T2, T3 = df.loc[train_mask, "incidence_per_100k"].quantile([0.50, 0.80, 0.95]).values
print(f"Tier thresholds (from 2014–2023 training data):")
print(f"  Low:      incidence < {T1:.2f}")
print(f"  Watch:    {T1:.2f} ≤ incidence < {T2:.2f}")
print(f"  Warning:  {T2:.2f} ≤ incidence < {T3:.2f}")
print(f"  Alert:    incidence ≥ {T3:.2f}")

def to_tier(inc):
    if inc < T1: return 0
    if inc < T2: return 1
    if inc < T3: return 2
    return 3

df["risk_tier_int"] = df["incidence_per_100k"].apply(to_tier).astype(int)

df["target_tier"]  = df.groupby("moh_name")["risk_tier_int"].shift(-1)
df["target_cases"] = df.groupby("moh_name")["cases"].shift(-1)

df = df.dropna(subset=["target_tier", "target_cases"]).reset_index(drop=True)
print(f"\nAfter shifting target by 1 week: {len(df):,} rows")

train = df[df["week_start"] < "2024-01-01"].copy()
val   = df[(df["week_start"] >= "2024-01-01") & (df["week_start"] < "2025-01-01")].copy()
test  = df[df["week_start"] >= "2025-01-01"].copy()

for name, part in [("Train 2014–2023", train), ("Val 2024", val), ("Test 2025–2026", test)]:
    mix = part["target_tier"].map(INT_TO_TIER).value_counts().reindex(TIER_ORDER).to_dict()
    print(f"  {name:<22s}: {len(part):>7,} rows  |  tier mix = {mix}")


Tier thresholds (from 2014–2023 training data):
  Low:      incidence < 2.75
  Watch:    2.75 ≤ incidence < 7.94
  Alert:    incidence ≥ 23.03

After shifting target by 1 week: 145,759 rows
  Train 2014–2023       : 118,198 rows  |  tier mix = {'Low': 58931, 'Watch': 35489, 'Warning': 17839, 'Alert': 5939}
  Val 2024              :  11,752 rows  |  tier mix = {'Low': 4730, 'Watch': 4965, 'Warning': 1862, 'Alert': 195}
  Test 2025–2026        :  15,809 rows  |  tier mix = {'Low': 5268, 'Watch': 6777, 'Warning': 3335, 'Alert': 429}


## 4. Feature Engineering (63 features)

All features are computed using data **≤ t** — there is no future leakage.

**Feature groups:**

1. **Case lags** at 1, 2, 3, 4, 5, 8, 12, 26, 52 weeks (52 = year-over-year)
2. **Rolling stats** — mean/max/std over 4/8/12 weeks (shifted by 1)
3. **Growth** — week-over-week growth, acceleration, 8-week linear trend
4. **Incidence** — same as case lags but expressed per 100k population
5. **Seasonality** — month + week-of-year, encoded as sin/cos
6. **District aggregates** — lagged district-level totals/means/max
7. **Spatial rank** — MOH's percentile rank and z-score within its district (last week)
8. **Weather** — current values + 4-week averages + engineered interactions
9. **Population** — raw and log-transformed population and density
10. **Outbreak recency** — weeks since cases > 20 in this MOH (capped at 52)
11. **Categorical** — `district_cat` (district encoded as integer 0…24)


In [4]:
g = df.groupby("moh_name")["cases"]

for lag in [1, 2, 3, 4, 5, 8, 12, 26, 52]:
    df[f"cases_lag{lag}"] = g.shift(lag)

_shifted = df.groupby("moh_name")["cases"].shift(1)
for window in [4, 8, 12]:
    df[f"cases_roll{window}_mean"] = _shifted.groupby(df["moh_name"]).rolling(window, min_periods=1).mean().reset_index(level=0, drop=True)
    df[f"cases_roll{window}_max"]  = _shifted.groupby(df["moh_name"]).rolling(window, min_periods=1).max().reset_index(level=0, drop=True)
    df[f"cases_roll{window}_std"]  = _shifted.groupby(df["moh_name"]).rolling(window, min_periods=1).std().reset_index(level=0, drop=True)

df["case_growth_wow"] = ((g.shift(1).values - g.shift(2).values) / (g.shift(2).values + 1.0)).clip(-5, 5)
df["case_accel"]      = df.groupby("moh_name")["case_growth_wow"].diff()

def _rolling_slope(s, window=8):
    def slope(arr):
        if len(arr) < 2 or np.std(arr) < 1e-6:
            return 0.0
        return float(np.polyfit(np.arange(len(arr)), arr, 1)[0])
    return s.rolling(window, min_periods=2).apply(slope, raw=True)
df["case_trend_8w"] = _shifted.groupby(df["moh_name"]).transform(lambda s: _rolling_slope(s, 8))

for lag in [1, 2, 4, 8]:
    df[f"inc_lag{lag}"] = df.groupby("moh_name")["incidence_per_100k"].shift(lag)
inc_shifted = df.groupby("moh_name")["incidence_per_100k"].shift(1)
for window in [4, 12]:
    df[f"inc_roll{window}_mean"] = inc_shifted.groupby(df["moh_name"]).rolling(window, min_periods=1).mean().reset_index(level=0, drop=True)

df["month"]     = df["week_start"].dt.month
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
df["woy"]       = df["week_start"].dt.isocalendar().week.astype(int)
df["woy_sin"]   = np.sin(2 * np.pi * df["woy"] / 52)
df["woy_cos"]   = np.cos(2 * np.pi * df["woy"] / 52)
df["iso_year"]  = df["week_start"].dt.isocalendar().year.astype(int)

district_week = (df.groupby(["district", "week_start"])
                 .agg(district_total=("cases", "sum"),
                      district_mean=("cases", "mean"),
                      district_max=("cases", "max"))
                 .reset_index().sort_values(["district", "week_start"]))
for lag in [1, 2, 4]:
    district_week[f"district_total_lag{lag}"] = district_week.groupby("district")["district_total"].shift(lag)
    district_week[f"district_mean_lag{lag}"]  = district_week.groupby("district")["district_mean"].shift(lag)
    district_week[f"district_max_lag{lag}"]   = district_week.groupby("district")["district_max"].shift(lag)
_shifted_total = district_week.groupby("district")["district_total"].shift(1)
district_week["district_total_roll4"]  = _shifted_total.groupby(district_week["district"]).rolling(4,  min_periods=1).mean().reset_index(level=0, drop=True)
district_week["district_total_roll12"] = _shifted_total.groupby(district_week["district"]).rolling(12, min_periods=1).mean().reset_index(level=0, drop=True)
df = df.merge(district_week[["district", "week_start",
                              "district_total_lag1", "district_total_lag2", "district_total_lag4",
                              "district_mean_lag1", "district_max_lag1",
                              "district_total_roll4", "district_total_roll12"]],
              on=["district", "week_start"], how="left")

prev_cases = df.groupby("moh_name")["cases"].shift(1)
df["_p"] = prev_cases
df["district_rank_lag1"]   = df.groupby(["district", "week_start"])["_p"].rank(pct=True)
df["district_zscore_lag1"] = ((df["_p"] - df.groupby(["district", "week_start"])["_p"].transform("mean")) /
                              (df.groupby(["district", "week_start"])["_p"].transform("std") + 1e-3))
df = df.drop(columns=["_p"])

df["temp_range"]   = df["temp_max"] - df["temp_min"]
df["rain_change"]  = df["rain_1w"] - df["rain_2w"] / 2.0
df["heat_index"]   = df["temp_avg"] * df["humidity"] / 100.0
df["rain_x_temp"]  = df["rain_1w"] * df["temp_avg"]
df["rain_x_humid"] = df["rain_1w"] * df["humidity"]

df["log_pop"]     = np.log1p(df["population"])
df["log_density"] = np.log1p(df["pop_density"])

def _wst(s, threshold=20):
    c, out = 999, []
    for v in s:
        if v > threshold: c = 0
        else: c += 1
        out.append(min(c, 52))
    return pd.Series(out, index=s.index)
df["_lc"] = df.groupby("moh_name")["cases"].shift(1)
df["weeks_since_outbreak_lag1"] = df.groupby("moh_name")["_lc"].transform(_wst)
df = df.drop(columns=["_lc"])

all_districts = sorted(df["district"].unique())
district_to_idx = {d: i for i, d in enumerate(all_districts)}
df["district_cat"] = df["district"].map(district_to_idx).astype("int32")

lag_cols = [c for c in df.columns
            if "lag" in c or "_roll" in c
            or c in ["case_trend_8w", "case_growth_wow", "case_accel"]]
df[lag_cols] = df.groupby("moh_name")[lag_cols].transform(lambda s: s.ffill().bfill()).fillna(0)
df["weeks_since_outbreak_lag1"] = df["weeks_since_outbreak_lag1"].fillna(52)

DROP_COLS = ["moh_code", "moh_name", "ds_name", "ds_id", "district", "week_start", "iso_week",
             "cases", "incidence_per_100k", "risk_tier", "risk_tier_int",
             "target_tier", "target_cases",
             "centroid_lat", "centroid_lon", "birth_rate", "area_km2"]
FEATURE_COLS = [c for c in df.columns if c not in DROP_COLS]
print(f" Built {len(FEATURE_COLS)} features")
print(f"   Sample: {FEATURE_COLS[:5]}  ...  {FEATURE_COLS[-3:]}")


 Built 63 features
   Sample: ['iso_year', 'cases_lag1', 'cases_lag2', 'temp_avg', 'temp_max']  ...  ['log_density', 'weeks_since_outbreak_lag1', 'district_cat']


In [5]:
train = df[df["week_start"] < "2024-01-01"].copy()
val   = df[(df["week_start"] >= "2024-01-01") & (df["week_start"] < "2025-01-01")].copy()
test  = df[df["week_start"] >= "2025-01-01"].copy()

for name, part in [("Train 2014–2023", train), ("Val 2024", val), ("Test 2025–2026", test)]:
    miss = [c for c in FEATURE_COLS if c not in part.columns]
    print(f"  {name:<22s}: {len(part):>7,} rows  |  missing features: {len(miss)}")
assert not any(c not in train.columns for c in FEATURE_COLS), "FE still missing on train"

  Train 2014–2023       : 118,198 rows  |  missing features: 0
  Val 2024              :  11,752 rows  |  missing features: 0
  Test 2025–2026        :  15,809 rows  |  missing features: 0


## 5. Train Base Models

We train **3 base models** in parallel for each of the two heads:

- **Tier head** (multi-class classification): LightGBM, XGBoost, CatBoost → stack with Logistic Regression
- **Case count head** (regression in log space): same 3 models → stack with Ridge

Class imbalance is handled via `class_weight="balanced"` (≈ up-weights Alert rows).
Early stopping on validation loss (patience 80 rounds) prevents overfitting.

**All hyperparameters are the same across heads** — they were tuned once on the val set and held fixed.

### 5.1 Risk Tier Classifier (target = next week's tier)

In [6]:
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import accuracy_score, f1_score, classification_report

X_train = train[FEATURE_COLS].fillna(0)
X_val   = val[FEATURE_COLS].fillna(0)
X_test  = test[FEATURE_COLS].fillna(0)
y_train = train["target_tier"].astype(int).values
y_val   = val["target_tier"].astype(int).values
y_test  = test["target_tier"].astype(int).values
sw      = compute_sample_weight("balanced", y_train)

print(f"Train: {X_train.shape}  |  Val: {X_val.shape}  |  Test: {X_test.shape}")
print(f"Target tier dist (test): {np.bincount(y_test, minlength=4)}")


Train: (118198, 63)  |  Val: (11752, 63)  |  Test: (15809, 63)
Target tier dist (test): [5268 6777 3335  429]


#### LightGBM (tier)

In [7]:
lgb_params = {
    "objective": "multiclass", "num_class": 4, "metric": "multi_logloss",
    "learning_rate": 0.05, "num_leaves": 63, "min_child_samples": 30,
    "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 5,
    "reg_alpha": 0.1, "reg_lambda": 0.5, "n_jobs": -1, "verbose": -1, "seed": RANDOM_SEED,
}
print("Training LightGBM (tier) ...")
dtrain = lgb.Dataset(X_train, y_train, weight=sw, categorical_feature=["district_cat"])
dval   = lgb.Dataset(X_val, y_val, reference=dtrain, categorical_feature=["district_cat"])
lgb_clf = lgb.train(lgb_params, dtrain, num_boost_round=2000,
                    valid_sets=[dval], valid_names=["val"],
                    callbacks=[lgb.early_stopping(80), lgb.log_evaluation(0)])
lgb_pred = lgb_clf.predict(X_test).argmax(axis=1)
lgb_acc = accuracy_score(y_test, lgb_pred)
print(f"LightGBM tier test accuracy: {lgb_acc:.4f}  (best iter: {lgb_clf.best_iteration})")


Training LightGBM (tier) ...
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[191]	val's multi_logloss: 0.689199
LightGBM tier test accuracy: 0.7191  (best iter: 191)


#### XGBoost (tier)

In [8]:
print("Training XGBoost (tier) ...")
xgb_clf = xgb.XGBClassifier(
    n_estimators=2000, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.5, min_child_weight=5,
    objective="multi:softprob", num_class=4, random_state=RANDOM_SEED, n_jobs=-1,
    eval_metric="mlogloss", early_stopping_rounds=80, tree_method="hist",
)
xgb_clf.fit(X_train, y_train, sample_weight=sw, eval_set=[(X_val, y_val)], verbose=0)
xgb_pred = xgb_clf.predict(X_test)
xgb_acc = accuracy_score(y_test, xgb_pred)
print(f" XGBoost tier test accuracy: {xgb_acc:.4f}  (best iter: {xgb_clf.best_iteration})")


Training XGBoost (tier) ...
 XGBoost tier test accuracy: 0.7270  (best iter: 342)


#### CatBoost (tier)

In [9]:
print("Training CatBoost (tier) — this is the slowest of the three ...")
cat_clf = CatBoostClassifier(
    iterations=2000, depth=7, learning_rate=0.05,
    loss_function="MultiClass", eval_metric="MultiClass",
    random_seed=RANDOM_SEED, verbose=0, early_stopping_rounds=80, l2_leaf_reg=3.0,
    cat_features=["district_cat"], task_type="CPU",
)
cat_clf.fit(X_train, y_train, sample_weight=sw,
            eval_set=(X_val, y_val), use_best_model=True)
cat_pred = cat_clf.predict(X_test).astype(int).ravel()
cat_acc = accuracy_score(y_test, cat_pred)
print(f" CatBoost tier test accuracy: {cat_acc:.4f}")


Training CatBoost (tier) — this is the slowest of the three ...
 CatBoost tier test accuracy: 0.7384


### 5.2 Stacking Ensemble (tier)

A Logistic Regression meta-learner combines the 3 base models' predicted class probabilities. The meta-learner is fit on the **validation** set (so the base models are out-of-fold) and applied to the test set for the final score.

In [10]:
from sklearn.linear_model import LogisticRegression

val_p3  = np.hstack([lgb_clf.predict(X_val),  xgb_clf.predict_proba(X_val),  cat_clf.predict_proba(X_val)])
test_p3 = np.hstack([lgb_clf.predict(X_test), xgb_clf.predict_proba(X_test), cat_clf.predict_proba(X_test)])

meta_clf = LogisticRegression(max_iter=2000, C=1.0, n_jobs=-1, random_state=RANDOM_SEED,
                              class_weight="balanced", solver="lbfgs")
meta_clf.fit(val_p3, y_val)
ens_pred = meta_clf.predict(test_p3)
ens_acc = accuracy_score(y_test, ens_pred)
print(f" Stacking ensemble tier test accuracy: {ens_acc:.4f}  (on {len(y_test):,} test rows)")

print("\nPer-class report (tier ensemble):")
print(classification_report(y_test, ens_pred, target_names=TIER_ORDER, digits=4))


 Stacking ensemble tier test accuracy: 0.7470  (on 15,809 test rows)

Per-class report (tier ensemble):
              precision    recall  f1-score   support

         Low     0.8013    0.7523    0.7760      5268
       Watch     0.7521    0.7351    0.7435      6777
     Warning     0.7024    0.7871    0.7424      3335
       Alert     0.4761    0.5571    0.5134       429

    accuracy                         0.7470     15809
   macro avg     0.6830    0.7079    0.6938     15809
weighted avg     0.7505    0.7470    0.7479     15809



### 5.3 Case Count Regressor (target = next week's cases)

In [11]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import Ridge
from catboost import CatBoostRegressor

y_train_c = train["target_cases"].values.astype(float)
y_val_c   = val["target_cases"].values.astype(float)
y_test_c  = test["target_cases"].values.astype(float)
y_train_log = np.log1p(y_train_c)
y_val_log   = np.log1p(y_val_c)

print(f"Case counts (test) — min {y_test_c.min():.0f}, max {y_test_c.max():.0f}, "
      f"mean {y_test_c.mean():.1f}, median {np.median(y_test_c):.0f}")


Case counts (test) — min 0, max 202, mean 4.8, median 2


#### LightGBM (cases)

In [12]:
print("Training LightGBM (cases) ...")
lgb_reg_params = {
    "objective": "regression", "metric": "mae",
    "learning_rate": 0.05, "num_leaves": 63, "min_child_samples": 30,
    "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 5,
    "reg_alpha": 0.1, "reg_lambda": 0.5, "n_jobs": -1, "verbose": -1, "seed": RANDOM_SEED,
}
dtrain_r = lgb.Dataset(X_train, y_train_log, categorical_feature=["district_cat"])
dval_r   = lgb.Dataset(X_val, y_val_log, reference=dtrain_r, categorical_feature=["district_cat"])
lgb_reg = lgb.train(lgb_reg_params, dtrain_r, num_boost_round=2000,
                    valid_sets=[dval_r], valid_names=["val"],
                    callbacks=[lgb.early_stopping(80), lgb.log_evaluation(0)])
lgb_reg_pred = np.expm1(lgb_reg.predict(X_test)).clip(0)
lgb_reg_mae  = mean_absolute_error(y_test_c, lgb_reg_pred)
print(f" LightGBM cases test MAE: {lgb_reg_mae:.2f}")


Training LightGBM (cases) ...
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[72]	val's l1: 0.248419
 LightGBM cases test MAE: 1.38


#### XGBoost (cases)

In [13]:
print("Training XGBoost (cases) ...")
xgb_reg = xgb.XGBRegressor(
    n_estimators=2000, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.5, min_child_weight=5,
    objective="reg:squarederror", random_state=RANDOM_SEED, n_jobs=-1,
    eval_metric="mae", early_stopping_rounds=80, tree_method="hist",
)
xgb_reg.fit(X_train, y_train_log, eval_set=[(X_val, y_val_log)], verbose=0)
xgb_reg_pred = np.expm1(xgb_reg.predict(X_test)).clip(0)
xgb_reg_mae  = mean_absolute_error(y_test_c, xgb_reg_pred)
print(f" XGBoost cases test MAE: {xgb_reg_mae:.2f}")


Training XGBoost (cases) ...
 XGBoost cases test MAE: 1.42


#### CatBoost (cases)

In [14]:
print("Training CatBoost (cases) ...")
cat_reg = CatBoostRegressor(
    iterations=2000, depth=7, learning_rate=0.05,
    loss_function="MAE", eval_metric="MAE",
    random_seed=RANDOM_SEED, verbose=0, early_stopping_rounds=80, l2_leaf_reg=3.0,
    cat_features=["district_cat"], task_type="CPU",
)
cat_reg.fit(X_train, y_train_log, eval_set=(X_val, y_val_log), use_best_model=True)
cat_reg_pred = np.expm1(cat_reg.predict(X_test)).clip(0)
cat_reg_mae  = mean_absolute_error(y_test_c, cat_reg_pred)
print(f" CatBoost cases test MAE: {cat_reg_mae:.2f}")


Training CatBoost (cases) ...
 CatBoost cases test MAE: 1.37


### 5.4 Stacking Ensemble (cases)

In [15]:
val_r3  = np.column_stack([lgb_reg.predict(X_val),  xgb_reg.predict(X_val),  cat_reg.predict(X_val)])
test_r3 = np.column_stack([lgb_reg.predict(X_test), xgb_reg.predict(X_test), cat_reg.predict(X_test)])

meta_reg = Ridge(alpha=1.0, random_state=RANDOM_SEED)
meta_reg.fit(val_r3, y_val_log)

ens_reg_pred = np.expm1(meta_reg.predict(test_r3)).clip(0)
ens_reg_mae  = mean_absolute_error(y_test_c, ens_reg_pred)
ens_reg_rmse = np.sqrt(mean_squared_error(y_test_c, ens_reg_pred))
_mask = y_test_c > 0
ens_reg_mape = np.mean(np.abs((y_test_c[_mask] - ens_reg_pred[_mask]) / y_test_c[_mask])) * 100
print(f" Stacking ensemble cases test MAE:  {ens_reg_mae:.2f}")
print(f"                              RMSE:  {ens_reg_rmse:.2f}")
print(f"                              MAPE:  {ens_reg_mape:.1f}%")


 Stacking ensemble cases test MAE:  1.35
                              RMSE:  3.31
                              MAPE:  34.4%


## 6. Performance Summary

Tier head is evaluated by accuracy + macro-F1. Case count head is evaluated by MAE / RMSE / MAPE. The naive persistence baseline uses the **current week's** tier/cases as the prediction for next week.

In [16]:
from sklearn.metrics import f1_score

tier_results = pd.DataFrame([
    ("Stacking Ensemble (3-model, prod)", ens_acc, f1_score(y_test, ens_pred, average="macro")),
    ("CatBoost",                         cat_acc, f1_score(y_test, cat_pred, average="macro")),
    ("LightGBM",                         lgb_acc, f1_score(y_test, lgb_pred, average="macro")),
    ("XGBoost",                          xgb_acc, f1_score(y_test, xgb_pred, average="macro")),
], columns=["model", "tier_accuracy", "tier_f1_macro"]).sort_values("tier_accuracy", ascending=False)

naive_tier_pred = test["risk_tier_int"].values
naive_tier_acc  = accuracy_score(y_test, naive_tier_pred)
naive_tier_f1   = f1_score(y_test, naive_tier_pred, average="macro")
tier_results = pd.concat([tier_results, pd.DataFrame([{
    "model": "Naive persistence (baseline)",
    "tier_accuracy": naive_tier_acc, "tier_f1_macro": naive_tier_f1,
}])], ignore_index=True).sort_values("tier_accuracy", ascending=False)

print("=" * 70)
print("TIER CLASSIFIER (next week)")
print("=" * 70)
print(tier_results.to_string(index=False))

naive_cases_pred = test["cases"].values
naive_reg_mae    = mean_absolute_error(y_test_c, naive_cases_pred)
reg_results = pd.DataFrame([
    ("Stacking Ensemble (3-model, prod)", ens_reg_mae, ens_reg_rmse, ens_reg_mape),
    ("CatBoost",                         cat_reg_mae,
     np.sqrt(mean_squared_error(y_test_c, cat_reg_pred)),
     np.mean(np.abs((y_test_c[_mask] - cat_reg_pred[_mask]) / y_test_c[_mask])) * 100),
    ("LightGBM",                         lgb_reg_mae,
     np.sqrt(mean_squared_error(y_test_c, lgb_reg_pred)),
     np.mean(np.abs((y_test_c[_mask] - lgb_reg_pred[_mask]) / y_test_c[_mask])) * 100),
    ("XGBoost",                          xgb_reg_mae,
     np.sqrt(mean_squared_error(y_test_c, xgb_reg_pred)),
     np.mean(np.abs((y_test_c[_mask] - xgb_reg_pred[_mask]) / y_test_c[_mask])) * 100),
    ("Naive persistence (baseline)",     naive_reg_mae,
     np.sqrt(mean_squared_error(y_test_c, naive_cases_pred)),
     np.mean(np.abs((y_test_c[_mask] - naive_cases_pred[_mask]) / y_test_c[_mask])) * 100),
], columns=["model", "mae", "rmse", "mape_pct"]).sort_values("mae")

print("\n" + "=" * 70)
print("CASE COUNT REGRESSOR (next week)")
print("=" * 70)
print(reg_results.to_string(index=False))


TIER CLASSIFIER (next week)
                            model  tier_accuracy  tier_f1_macro
     Naive persistence (baseline)       0.754633       0.751294
Stacking Ensemble (3-model, prod)       0.746980       0.693828
                         CatBoost       0.738440       0.688236
                          XGBoost       0.726991       0.680232
                         LightGBM       0.719084       0.678951

CASE COUNT REGRESSOR (next week)
                            model      mae     rmse  mape_pct
     Naive persistence (baseline) 1.242077 3.227025 36.339803
Stacking Ensemble (3-model, prod) 1.349530 3.311030 34.352635
                         CatBoost 1.365485 3.566275 33.390666
                         LightGBM 1.380369 3.381495 35.163836
                          XGBoost 1.419046 3.501126 35.083146


## 7. Save Artifacts

Everything in `models/` is what the web app loads at startup. `pipeline_meta.json` is the contract — read it first if anything looks weird.

In [17]:
print("Saving artifacts to", MODEL_DIR, "...")

lgb_clf.save_model(f"{MODEL_DIR}/lgb_classifier.txt")
xgb_clf.save_model(f"{MODEL_DIR}/xgb_classifier.json")
cat_clf.save_model(f"{MODEL_DIR}/cat_classifier.cbm")

lgb_reg.save_model(f"{MODEL_DIR}/lgb_regressor.txt")
xgb_reg.save_model(f"{MODEL_DIR}/xgb_regressor.json")
cat_reg.save_model(f"{MODEL_DIR}/cat_regressor.cbm")

with open(f"{MODEL_DIR}/meta_classifier.pkl", "wb") as f: pickle.dump(meta_clf, f)
with open(f"{MODEL_DIR}/meta_regressor.pkl",  "wb") as f: pickle.dump(meta_reg, f)

pipeline_meta = {
    "task": "1-week-ahead forecast (predict week t+1 from features at week t)",
    "feature_cols": FEATURE_COLS,
    "categorical_cols": ["district_cat"],
    "tier_to_int": TIER_TO_INT,
    "int_to_tier": {str(k): v for k, v in INT_TO_TIER.items()},
    "tier_thresholds": [float(T1), float(T2), float(T3)],
    "alert_decision_threshold": ALERT_THRESHOLD,
    "random_seed": RANDOM_SEED,
    "district_to_idx": district_to_idx,
    "case_count_log_transform": True,
    "models": {
        "tier_classifier":   ["lgb_classifier.txt", "xgb_classifier.json", "cat_classifier.cbm", "meta_classifier.pkl"],
        "case_regressor":    ["lgb_regressor.txt",  "xgb_regressor.json",  "cat_regressor.cbm",  "meta_regressor.pkl"],
    },
}
with open(f"{MODEL_DIR}/pipeline_meta.json", "w") as f:
    json.dump(pipeline_meta, f, indent=2)

print(f"\n Saved to {MODEL_DIR}/:")
for f in sorted(os.listdir(MODEL_DIR)):
    size = os.path.getsize(f"{MODEL_DIR}/{f}") / 1024
    print(f"   {f:<28s}  {size:>8.1f} KB")


Saving artifacts to models ...

 Saved to models/:
   cat_classifier.cbm              1304.5 KB
   cat_regressor.cbm                361.4 KB
   lgb_classifier.txt              5130.6 KB
   lgb_regressor.txt                414.9 KB
   meta_classifier.pkl                1.1 KB
   meta_regressor.pkl                 0.5 KB
   pipeline_meta.json                 2.6 KB
   xgb_classifier.json             9849.3 KB
   xgb_regressor.json              1023.9 KB


## 8. Production Predictor

`DengueRadarPredictor` is the **only** class the web app needs to import. It loads artifacts, takes a DataFrame of weekly features for every MOH, and returns next-week tier + case-count predictions.

**Input contract:**
- One row per (MOH, current week)
- Must contain all 63 columns listed in `FEATURE_COLS` (or a subset; missing ones are filled with 0)
- `district_cat` is required as the encoded district (see `district_to_idx` in `pipeline_meta.json`)

**Output:**
- `predict_week(week_start, frame)` → DataFrame with: `moh_name`, `district`, `predicted_tier`, `predicted_cases`, `p_Low`, `p_Watch`, `p_Warning`, `p_Alert`, `alert_high_confidence`

In [18]:
class DengueRadarPredictor:
    """Loads the trained artifacts and produces 1-week-ahead dengue forecasts."""

    def __init__(self, model_dir="models"):
        import lightgbm as lgb
        import xgboost as xgb
        from catboost import CatBoostClassifier, CatBoostRegressor

        self.lgb = lgb.Booster(model_file=f"{model_dir}/lgb_classifier.txt")
        self.xgb = xgb.XGBClassifier(); self.xgb.load_model(f"{model_dir}/xgb_classifier.json")
        self.cat = CatBoostClassifier(); self.cat.load_model(f"{model_dir}/cat_classifier.cbm")
        with open(f"{model_dir}/meta_classifier.pkl", "rb") as f:
            self.meta = pickle.load(f)

        self.lgb_reg = lgb.Booster(model_file=f"{model_dir}/lgb_regressor.txt")
        self.xgb_reg = xgb.XGBRegressor(); self.xgb_reg.load_model(f"{model_dir}/xgb_regressor.json")
        self.cat_reg = CatBoostRegressor(); self.cat_reg.load_model(f"{model_dir}/cat_regressor.cbm")
        with open(f"{model_dir}/meta_regressor.pkl", "rb") as f:
            self.meta_reg = pickle.load(f)

        with open(f"{model_dir}/pipeline_meta.json") as f:
            self.meta_json = json.load(f)
        self.feature_cols     = self.meta_json["feature_cols"]
        self.int_to_tier      = {int(k): v for k, v in self.meta_json["int_to_tier"].items()}
        self.alert_threshold  = self.meta_json.get("alert_decision_threshold", 0.5)
        print(f" DengueRadarPredictor loaded ({len(self.feature_cols)} features)")

    def _stack_tier(self, X):
        return np.hstack([self.lgb.predict(X),
                          self.xgb.predict_proba(X),
                          self.cat.predict_proba(X)])

    def _stack_cases(self, X):
        return np.column_stack([self.lgb_reg.predict(X),
                                self.xgb_reg.predict(X),
                                self.cat_reg.predict(X)])

    def predict(self, df):
        """Return (tier_int, probs). tier_int is 0..3."""
        X = df[self.feature_cols].fillna(0)
        probs = self.meta.predict_proba(self._stack_tier(X))
        return probs.argmax(axis=1), probs

    def predict_cases(self, df):
        """Return predicted next-week case counts (float, ≥ 0)."""
        X = df[self.feature_cols].fillna(0)
        return np.expm1(self.meta_reg.predict(self._stack_cases(X))).clip(0)

    def predict_week(self, week_start, frame):
        """
        Predict the next week (week_start + 7 days) for every MOH row matching `week_start`.
        `frame` must already have all 63 feature columns computed for week `week_start`.
        """
        rows = frame[frame["week_start"] == pd.Timestamp(week_start)]
        if rows.empty:
            return pd.DataFrame()
        pred, probs = self.predict(rows)
        cases = self.predict_cases(rows)
        out = rows[["moh_name", "district"]].copy().reset_index(drop=True)
        out["predicted_tier"]       = [self.int_to_tier[p] for p in pred]
        out["predicted_cases"]      = np.round(cases).astype(int)
        out[["p_Low", "p_Watch", "p_Warning", "p_Alert"]] = probs
        out["alert_high_confidence"] = out["p_Alert"] > self.alert_threshold
        out["action_priority"]      = (out["p_Alert"] * np.log1p(out["predicted_cases"])).round(3)
        out["target_week_start"]    = pd.Timestamp(week_start) + pd.Timedelta(days=7)
        return out.sort_values("p_Alert", ascending=False).reset_index(drop=True)

predictor_source = '''\
"""dengueradar_predictor.py — production inference for the dengue forecast model."""
import json, pickle
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, CatBoostRegressor


class DengueRadarPredictor:
    def __init__(self, model_dir="models"):
        self.lgb = lgb.Booster(model_file=f"{model_dir}/lgb_classifier.txt")
        self.xgb = xgb.XGBClassifier(); self.xgb.load_model(f"{model_dir}/xgb_classifier.json")
        self.cat = CatBoostClassifier(); self.cat.load_model(f"{model_dir}/cat_classifier.cbm")
        with open(f"{model_dir}/meta_classifier.pkl", "rb") as f:
            self.meta = pickle.load(f)

        self.lgb_reg = lgb.Booster(model_file=f"{model_dir}/lgb_regressor.txt")
        self.xgb_reg = xgb.XGBRegressor(); self.xgb_reg.load_model(f"{model_dir}/xgb_regressor.json")
        self.cat_reg = CatBoostRegressor(); self.cat_reg.load_model(f"{model_dir}/cat_regressor.cbm")
        with open(f"{model_dir}/meta_regressor.pkl", "rb") as f:
            self.meta_reg = pickle.load(f)

        with open(f"{model_dir}/pipeline_meta.json") as f:
            self.meta_json = json.load(f)
        self.feature_cols    = self.meta_json["feature_cols"]
        self.int_to_tier     = {int(k): v for k, v in self.meta_json["int_to_tier"].items()}
        self.alert_threshold = self.meta_json.get("alert_decision_threshold", 0.5)

    def _stack_tier(self, X):
        return np.hstack([self.lgb.predict(X),
                          self.xgb.predict_proba(X),
                          self.cat.predict_proba(X)])

    def _stack_cases(self, X):
        return np.column_stack([self.lgb_reg.predict(X),
                                self.xgb_reg.predict(X),
                                self.cat_reg.predict(X)])

    def predict(self, df):
        X = df[self.feature_cols].fillna(0)
        probs = self.meta.predict_proba(self._stack_tier(X))
        return probs.argmax(axis=1), probs

    def predict_cases(self, df):
        X = df[self.feature_cols].fillna(0)
        return np.expm1(self.meta_reg.predict(self._stack_cases(X))).clip(0)

    def predict_week(self, week_start, frame):
        rows = frame[frame["week_start"] == pd.Timestamp(week_start)]
        if rows.empty:
            return pd.DataFrame()
        pred, probs = self.predict(rows)
        cases = self.predict_cases(rows)
        out = rows[["moh_name", "district"]].copy().reset_index(drop=True)
        out["predicted_tier"]        = [self.int_to_tier[p] for p in pred]
        out["predicted_cases"]       = np.round(cases).astype(int)
        out[["p_Low","p_Watch","p_Warning","p_Alert"]] = probs
        out["alert_high_confidence"] = out["p_Alert"] > self.alert_threshold
        out["action_priority"]       = (out["p_Alert"] * np.log1p(out["predicted_cases"])).round(3)
        out["target_week_start"]     = pd.Timestamp(week_start) + pd.Timedelta(days=7)
        return out.sort_values("p_Alert", ascending=False).reset_index(drop=True)
'''
with open("dengueradar_predictor.py", "w") as f:
    f.write(predictor_source)
print(" Wrote dengueradar_predictor.py — drop it into your web-app repo.")


 Wrote dengueradar_predictor.py — drop it into your web-app repo.


## 9. Inference Demo

Run `predict_week` on the test set and inspect the output. We pick the latest week so the demo shows what the cron job will produce every Monday morning.

In [19]:
predictor = DengueRadarPredictor(MODEL_DIR)

latest_week = test["week_start"].max()
print(f"\n=== 1-week-ahead forecast issued on {latest_week.date()} ===")
print(f"    (features from week of {latest_week.date()} → predicting week of "
      f"{(latest_week + pd.Timedelta(days=7)).date()})")

weekly = predictor.predict_week(latest_week, test)
print(f"\nPredictions for {len(weekly)} MOHs")
print(f"Tier breakdown:  {weekly['predicted_tier'].value_counts().reindex(TIER_ORDER).fillna(0).astype(int).to_dict()}")
print(f"Total predicted cases (next week): {weekly['predicted_cases'].sum():,}")
print(f"\nTop 15 MOHs by Alert probability:")
print(weekly.head(15)[
    ["moh_name", "district", "predicted_tier", "predicted_cases", "p_Alert", "alert_high_confidence"]
].to_string(index=False))


 DengueRadarPredictor loaded (63 features)

=== 1-week-ahead forecast issued on 2026-05-04 ===
    (features from week of 2026-05-04 → predicting week of 2026-05-11)

Predictions for 226 MOHs
Tier breakdown:  {'Low': 52, 'Watch': 62, 'Warning': 109, 'Alert': 3}
Total predicted cases (next week): 1,245

Top 15 MOHs by Alert probability:
            moh_name    district predicted_tier  predicted_cases  p_Alert  alert_high_confidence
              Eravur  Batticaloa          Alert               26 0.991149                   True
              Nallur      Jaffna          Alert                6 0.838673                   True
           Mc jaffna      Jaffna          Alert               13 0.750164                   True
           Poonakary Kilinochchi        Warning                1 0.289704                  False
            Moratuwa     Colombo        Warning               25 0.270749                  False
          MC Colombo     Colombo        Warning               94 0.269174       

## 10. Web App Integration (FastAPI)

Below is a **drop-in FastAPI service** that loads the model once at startup and exposes two endpoints:

| Method | Path | Purpose |
|---|---|---|
| `GET`  | `/health` | Liveness check — returns `{"status": "ok"}` |
| `POST` | `/predict/next-week` | Predict tier + case count for next week |

**Request body** (`POST /predict/next-week`):

```json
{
  "feature_week_start": "2025-06-02",
  "rows": [
    {"moh_name": "Colombo MC", "district": "Colombo", "week_start": "2025-06-02", "cases_lag1": 12, ...},
    ...
  ]
}
```

**Response body:**

```json
{
  "feature_week_start": "2025-06-02",
  "target_week_start": "2025-06-09",
  "predictions": [
    {"moh_name": "Colombo MC", "district": "Colombo", "predicted_tier": "Alert",
     "predicted_cases": 24, "p_Alert": 0.71, "alert_high_confidence": true, ...},
    ...
  ]
}
```

The cell below writes `app.py` so you can run it with `uvicorn app:app --host 0.0.0.0 --port 8000`.

In [20]:
app_source = '''\
"""app.py — FastAPI service that serves the 1-week-ahead dengue forecast.

Run:
    pip install fastapi uvicorn[standard] pydantic lightgbm xgboost catboost pandas numpy
    uvicorn app:app --host 0.0.0.0 --port 8000

The model artifacts in ./models/ are loaded once at startup. The /predict/next-week endpoint
expects a list of rows that already have all 63 feature columns. If your pipeline computes the
features upstream, just POST the result here.
"""
from typing import List, Optional
from datetime import date

import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

from dengueradar_predictor import DengueRadarPredictor

app = FastAPI(title="DengueRadar Forecast API", version="1.0.0")
predictor: Optional[DengueRadarPredictor] = None


class FeatureRow(BaseModel):
    """One row of input features for a (MOH, current week) pair.

    Must include all 63 feature columns. See pipeline_meta.json for the full list.
    District must be passed as district_cat (int) — encode it on your side using the
    district_to_idx map in pipeline_meta.json.
    """
    moh_name: str
    district: str
    week_start: date
    # the 63 numeric features are accepted as a free-form dict so the schema is forward-compatible
    features: dict = Field(default_factory=dict)


class PredictRequest(BaseModel):
    feature_week_start: date
    rows: List[FeatureRow]


class Prediction(BaseModel):
    moh_name: str
    district: str
    predicted_tier: str
    predicted_cases: int
    p_Low: float
    p_Watch: float
    p_Warning: float
    p_Alert: float
    alert_high_confidence: bool
    action_priority: float


class PredictResponse(BaseModel):
    feature_week_start: date
    target_week_start: date
    n_predictions: int
    predictions: List[Prediction]


@app.on_event("startup")
def _load():
    global predictor
    predictor = DengueRadarPredictor("models")


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/predict/next-week", response_model=PredictResponse)
def predict_next_week(req: PredictRequest):
    if not req.rows:
        raise HTTPException(status_code=400, detail="rows is empty")
    if predictor is None:
        raise HTTPException(status_code=503, detail="model not loaded")

    # Build a DataFrame with one row per (MOH, week_start). All 63 features must be present
    # in the `features` dict of each row.
    feature_cols = predictor.feature_cols
    records = []
    for r in req.rows:
        row = {"moh_name": r.moh_name, "district": r.district, "week_start": pd.Timestamp(r.week_start)}
        for col in feature_cols:
            row[col] = r.features.get(col, 0.0)
        records.append(row)
    df = pd.DataFrame(records)

    out = predictor.predict_week(req.feature_week_start, df)
    if out.empty:
        raise HTTPException(status_code=404, detail="no rows match the requested feature_week_start")

    target_week = (pd.Timestamp(req.feature_week_start) + pd.Timedelta(days=7)).date()
    preds = [
        Prediction(
            moh_name=str(r["moh_name"]),
            district=str(r["district"]),
            predicted_tier=str(r["predicted_tier"]),
            predicted_cases=int(r["predicted_cases"]),
            p_Low=float(r["p_Low"]),
            p_Watch=float(r["p_Watch"]),
            p_Warning=float(r["p_Warning"]),
            p_Alert=float(r["p_Alert"]),
            alert_high_confidence=bool(r["alert_high_confidence"]),
            action_priority=float(r["action_priority"]),
        )
        for _, r in out.iterrows()
    ]
    return PredictResponse(
        feature_week_start=req.feature_week_start,
        target_week_start=target_week,
        n_predictions=len(preds),
        predictions=preds,
    )
'''
with open("app.py", "w") as f:
    f.write(app_source)
print(" Wrote app.py — FastAPI service for /predict/next-week")
print("   Run with: uvicorn app:app --host 0.0.0.0 --port 8000")


 Wrote app.py — FastAPI service for /predict/next-week
   Run with: uvicorn app:app --host 0.0.0.0 --port 8000


## 11. Deployment Checklist

### Inputs the web app needs to provide

For every (MOH, current week) pair, the upstream pipeline must produce **63 numeric features** in a single row. The full list is in `models/pipeline_meta.json → feature_cols`. Quick reference:

| Group | Features |
|---|---|
| Case lags | `cases_lag{1,2,3,4,5,8,12,26,52}` |
| Case rolling | `cases_roll{4,8,12}_{mean,max,std}` |
| Growth | `case_growth_wow`, `case_accel`, `case_trend_8w` |
| Incidence | `inc_lag{1,2,4,8}`, `inc_roll{4,12}_mean` |
| Seasonality | `iso_year`, `month`, `month_sin/cos`, `woy`, `woy_sin/cos` |
| District | `district_total_lag{1,2,4}`, `district_mean_lag1`, `district_max_lag1`, `district_total_roll{4,12}` |
| Spatial rank | `district_rank_lag1`, `district_zscore_lag1` |
| Weather | `temp_avg/max/min`, `temp_range`, `temp_avg_4w`, `humidity`, `humidity_4w`, `rain_1w/2w/4w`, `rain_change`, `heat_index`, `rain_x_temp`, `rain_x_humid` |
| Population | `population`, `pop_density`, `log_pop`, `log_density` |
| Other | `weeks_since_outbreak_lag1` |
| Categorical | `district_cat` (int 0–24, see `district_to_idx` in `pipeline_meta.json`) |

### What the model outputs

- **`predicted_tier`** ∈ {`Low`, `Watch`, `Warning`, `Alert`} — for week `t+1`
- **`predicted_cases`** — integer ≥ 0, expected case count for week `t+1`
- **`p_Alert`** — probability of Alert; `alert_high_confidence` = (p_Alert > 0.5)
- **`action_priority`** = `p_Alert * log1p(predicted_cases)` — sort by this for the alert queue

### Retraining schedule

- **Weekly:** the cron job that runs the predictor does **not** retrain. It only loads artifacts and predicts.
- **Quarterly** (or whenever a major outbreak pattern shifts): run this notebook end-to-end with the latest data to refresh the artifacts. Re-deploy the new `models/` directory.

### Files in `models/`

```
lgb_classifier.txt   xgb_classifier.json   cat_classifier.cbm   meta_classifier.pkl
lgb_regressor.txt    xgb_regressor.json    cat_regressor.cbm    meta_regressor.pkl
pipeline_meta.json   # read this first if anything looks weird
```

### Production cron job (sketch)

```bash
# 1. Every Monday at 06:00 — pull latest weather + last 52 weeks of cases
# 2. Compute the 63 features for all 226 MOHs (re-use the feature engineering cells above)
# 3. POST to http://localhost:8000/predict/next-week
# 4. Filter to alert_high_confidence == True and notify (WhatsApp / SMS / dashboard)
```

# **Final Test**

In [21]:
from sklearn.metrics import confusion_matrix, classification_report

N_MOH_TOTAL = df["moh_name"].nunique()
print(f"Total MOHs in dataset: {N_MOH_TOTAL}")

all_test_weeks = sorted(test["week_start"].unique(), reverse=True)
latest_week = None
for w in all_test_weeks:
    cov = test.loc[test["week_start"] == w, "moh_name"].nunique()
    if cov == N_MOH_TOTAL:
        latest_week = w
        break
if latest_week is None:
    latest_week = all_test_weeks[0]
    cov = test.loc[test["week_start"] == latest_week, "moh_name"].nunique()
    print(f"  No week has all {N_MOH_TOTAL} MOHs; using {latest_week.date()} ({cov} MOHs)")
else:
    print(f" Using {latest_week.date()} — full {N_MOH_TOTAL} MOH coverage")

target_week = latest_week + pd.Timedelta(days=7)

preds_all = predictor.predict_week(latest_week, test)
print(f"\nFeatures from:  {latest_week.date()}")
print(f"Predicting for: {target_week.date()}")
print(f"Predictions generated for: {len(preds_all)} MOHs")


print("\n" + "=" * 70)
print("STATUS BREAKDOWN — all MOHs")
print("=" * 70)
tier_counts = (preds_all["predicted_tier"]
               .value_counts()
               .reindex(TIER_ORDER)
               .fillna(0)
               .astype(int))
print(tier_counts.to_string())
print(f"\nTotal: {tier_counts.sum()}")
for t in TIER_ORDER:
    pct = 100 * tier_counts[t] / max(tier_counts.sum(), 1)
    print(f"  {t:<8s} {tier_counts[t]:>4d}  ({pct:5.1f}%)")

print("\n" + "=" * 70)
print("PREDICTED vs ACTUAL — same week")
print("=" * 70)
truth = (test.loc[test["week_start"] == latest_week, ["moh_name", "target_tier"]]
         .rename(columns={"target_tier": "actual_tier_int"}))
truth["actual_tier"] = truth["actual_tier_int"].map(INT_TO_TIER)
merged = preds_all.merge(truth[["moh_name", "actual_tier", "actual_tier_int"]],
                         on="moh_name", how="left")
n_with_truth = merged["actual_tier"].notna().sum()
print(f"MOHs with ground truth available: {n_with_truth} / {len(merged)}")

actual_counts = (merged["actual_tier"]
                 .value_counts()
                 .reindex(TIER_ORDER)
                 .fillna(0)
                 .astype(int))
print(f"\n  Actual    : {actual_counts.to_dict()}")
print(f"  Predicted : {tier_counts.to_dict()}")


mask_eval = merged["actual_tier_int"].notna()
y_true = merged.loc[mask_eval, "actual_tier_int"].astype(int).values
y_pred = (merged.loc[mask_eval, "predicted_tier"]
          .map(TIER_TO_INT).astype(int).values)
acc = accuracy_score(y_true, y_pred)
f1m = f1_score(y_true, y_pred, average="macro")
print(f"\nLatest-week accuracy: {acc:.4f}  |  macro-F1: {f1m:.4f}")

cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3])
print("\nConfusion matrix (rows = true, cols = pred):")
print(pd.DataFrame(cm, index=TIER_ORDER, columns=TIER_ORDER))

print("\n" + classification_report(y_true, y_pred,
                                  target_names=TIER_ORDER, digits=3))


n_alerts_hc       = int(preds_all["alert_high_confidence"].sum())
n_warn_or_alert   = int(preds_all["predicted_tier"].isin(["Warning", "Alert"]).sum())
n_watch_or_higher = int(preds_all["predicted_tier"].isin(["Watch", "Warning", "Alert"]).sum())
print("=" * 70)
print("ALERT BREAKDOWN")
print("=" * 70)
print(f"  High-confidence Alerts (p_Alert > 0.5) : {n_alerts_hc}")
print(f"  Warning or Alert (any confidence)      : {n_warn_or_alert}")
print(f"  Watch or higher                        : {n_watch_or_higher}")


print("\n" + "=" * 70)
print("PER-DISTRICT FORECAST")
print("=" * 70)
top_per_district = (preds_all.sort_values("p_Alert", ascending=False)
                    .groupby("district").first()["moh_name"])
district_breakdown = (preds_all
    .groupby("district")
    .agg(n_moh=("moh_name", "count"),
         n_alert=("alert_high_confidence", "sum"),
         total_cases_next_week=("predicted_cases", "sum"),
         mean_p_alert=("p_Alert", "mean"),
         mean_cases=("predicted_cases", "mean"))
    .join(top_per_district.rename("top_alert_moh"))
    .sort_values(["n_alert", "mean_p_alert"], ascending=False))
print(district_breakdown.to_string())


print("\n" + "=" * 70)
print("PREDICTED CASES — NEXT WEEK")
print("=" * 70)
print(f"  Total cases (all MOHs)  : {int(preds_all['predicted_cases'].sum()):,}")
print(f"  Mean per MOH            : {preds_all['predicted_cases'].mean():.1f}")
print(f"  Median per MOH          : {preds_all['predicted_cases'].median():.0f}")
print(f"  Max                     : {int(preds_all['predicted_cases'].max())}")
print(f"  Min                     : {int(preds_all['predicted_cases'].min())}")
print(f"  MOHs with 0 cases       : {(preds_all['predicted_cases'] == 0).sum()}")


print("\n" + "=" * 70)
print("PROBABILITY DISTRIBUTION (each row should sum to 1.0)")
print("=" * 70)
for c in ["p_Low", "p_Watch", "p_Warning", "p_Alert"]:
    print(f"  {c:<10s} mean={preds_all[c].mean():.3f}  "
          f"max={preds_all[c].max():.3f}  min={preds_all[c].min():.3f}")
prob_sum = preds_all[["p_Low", "p_Watch", "p_Warning", "p_Alert"]].sum(axis=1)
print(f"  Row-sum mean: {prob_sum.mean():.4f}  (should be ≈1.0)")


print("\n" + "=" * 70)
print("TOP 20 MOHs BY ACTION PRIORITY  (p_Alert × log1p(predicted_cases))")
print("=" * 70)
top20 = preds_all.head(20)[["moh_name", "district", "predicted_tier",
                            "predicted_cases", "p_Alert",
                            "alert_high_confidence", "action_priority"]]
print(top20.to_string(index=False))


print("\n" + "=" * 70)
print("COVERAGE CHECK")
print("=" * 70)
all_mohs        = set(df["moh_name"].unique())
predicted_mohs  = set(preds_all["moh_name"].unique())
missing         = sorted(all_mohs - predicted_mohs)
extra           = sorted(predicted_mohs - all_mohs)
print(f"  All MOHs in dataset : {len(all_mohs)}")
print(f"  Predicted this week : {len(predicted_mohs)}")
if missing:
    print(f"    Missing: {len(missing)} MOHs — sample: {missing[:10]}")
if extra:
    print(f"    Unexpected: {len(extra)} MOHs — sample: {extra[:10]}")
if not missing and not extra:
    print(f"   Perfect coverage — all {N_MOH_TOTAL} MOHs predicted.")



Total MOHs in dataset: 226
 Using 2026-05-04 — full 226 MOH coverage

Features from:  2026-05-04
Predicting for: 2026-05-11
Predictions generated for: 226 MOHs

STATUS BREAKDOWN — all MOHs
predicted_tier
Low         52
Watch       62
Warning    109
Alert        3

Total: 226
  Low        52  ( 23.0%)
  Watch      62  ( 27.4%)
  Warning   109  ( 48.2%)
  Alert       3  (  1.3%)

PREDICTED vs ACTUAL — same week
MOHs with ground truth available: 226 / 226

  Actual    : {'Low': 54, 'Watch': 65, 'Warning': 106, 'Alert': 1}
  Predicted : {'Low': 52, 'Watch': 62, 'Warning': 109, 'Alert': 3}

Latest-week accuracy: 0.8628  |  macro-F1: 0.7524

Confusion matrix (rows = true, cols = pred):
         Low  Watch  Warning  Alert
Low       40     12        2      0
Watch     12     50        3      0
Warning    0      0      104      2
Alert      0      0        0      1

              precision    recall  f1-score   support

         Low      0.769     0.741     0.755        54
       Watch      0.8